# Notebook 05 — Construcción de `silver.ventas_minoristas`

## Objetivo

Construir la tabla `silver.ventas_minoristas` a partir de `bronze.ventas_minoristas` aplicando los siguientes criterios:

1. **Ventana temporal**: filtrar el rango 2022-01-01 a 2025-12-31, eliminando años incompletos (2019, 2020, 2021, 2026) y errores de carga (2028).
2. **Tipos de datos**: castear `id_cliente` a STRING para garantizar coherencia con `dim_cliente` y `fact_lineas_pedido`.
3. **Identificador de producto**: crear `id_sku` derivado de la concatenación de modelo, color y talla.
4. **Normalización de texto**: aplicar `TRIM` y `UPPER` a campos categóricos (canal_venta, modelo, color, talla).
5. **Validaciones**: comprobar volumen final, integridad referencial con `dim_cliente` y distribución por canal.

## 1. Configuración y conexión a DuckDB

In [1]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")
print(f"Tamaño actual: {RUTA_DUCKDB.stat().st_size / (1024*1024):.2f} MB")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Tamaño actual: 66.51 MB


## 2. Inspección previa de `bronze.ventas_minoristas`

Antes de construir Silver, revisamos el estado de partida: volumen, esquema, distribución temporal, canales y calidad de fechas.

In [2]:
print("=" * 60)
print("ESQUEMA DE bronze.ventas_minoristas")
print("=" * 60)
esquema = con.execute("DESCRIBE bronze.ventas_minoristas").fetchdf()
print(esquema.to_string(index=False))

print("\n" + "=" * 60)
print("VOLUMEN TOTAL")
print("=" * 60)
total = con.execute("SELECT COUNT(*) FROM bronze.ventas_minoristas").fetchone()[0]
print(f"Filas totales: {total:,}")

ESQUEMA DE bronze.ventas_minoristas
                     column_name column_type null  key default extra
                     fecha_venta        DATE  YES None    None  None
                      cod_modelo     VARCHAR  YES None    None  None
                     desc_modelo     VARCHAR  YES None    None  None
                       cod_color     VARCHAR  YES None    None  None
                      desc_color     VARCHAR  YES None    None  None
                       cod_serie     VARCHAR  YES None    None  None
                      desc_serie     VARCHAR  YES None    None  None
                           talla     VARCHAR  YES None    None  None
                   cantidad_neta      BIGINT  YES None    None  None
cantidad_ventas_sin_devoluciones      BIGINT  YES None    None  None
                      id_cliente      BIGINT  YES None    None  None
                  nombre_cliente     VARCHAR  YES None    None  None
                  id_tipo_pedido     VARCHAR  YES None    None  Non

In [3]:
print("=" * 60)
print("DISTRIBUCIÓN POR AÑO")
print("=" * 60)
por_anio = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes_distintos,
        MIN(fecha_venta) AS fecha_min,
        MAX(fecha_venta) AS fecha_max
    FROM bronze.ventas_minoristas
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(por_anio.to_string(index=False))

DISTRIBUCIÓN POR AÑO
 anio  filas  clientes_distintos  fecha_min  fecha_max
 2022 729665                2266 2022-01-01 2022-12-31
 2023 719482                2232 2023-01-01 2023-12-31
 2024 705210                2200 2024-01-01 2024-12-31
 2025 716213                2125 2025-01-01 2025-12-31


In [5]:
print("=" * 60)
print("LISTADO COMPLETO DE COLUMNAS DE bronze.ventas_minoristas")
print("=" * 60)
columnas = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze' 
      AND table_name = 'ventas_minoristas'
    ORDER BY ordinal_position
""").fetchdf()
print(columnas.to_string(index=False))

LISTADO COMPLETO DE COLUMNAS DE bronze.ventas_minoristas
                     column_name data_type
                     fecha_venta      DATE
                      cod_modelo   VARCHAR
                     desc_modelo   VARCHAR
                       cod_color   VARCHAR
                      desc_color   VARCHAR
                       cod_serie   VARCHAR
                      desc_serie   VARCHAR
                           talla   VARCHAR
                   cantidad_neta    BIGINT
cantidad_ventas_sin_devoluciones    BIGINT
                      id_cliente    BIGINT
                  nombre_cliente   VARCHAR
                  id_tipo_pedido   VARCHAR
                desc_tipo_pedido   VARCHAR
                    id_temporada    BIGINT


In [6]:
print("=" * 60)
print("DISTRIBUCIÓN POR TIPO DE PEDIDO (CANAL)")
print("=" * 60)
por_tipo = con.execute("""
    SELECT 
        id_tipo_pedido,
        desc_tipo_pedido,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes_distintos
    FROM bronze.ventas_minoristas
    GROUP BY id_tipo_pedido, desc_tipo_pedido
    ORDER BY filas DESC
""").fetchdf()
print(por_tipo.to_string(index=False))

DISTRIBUCIÓN POR TIPO DE PEDIDO (CANAL)
id_tipo_pedido   desc_tipo_pedido   filas  clientes_distintos
             1          Temporada 1606006                2534
             2         Repetición  615079                2584
             7           Depósito  140135                   6
          eCom          Ecommerce  111233                   5
         77/88                ECI   92678                   8
           B2B                B2B   82940                 455
             5           Muestras   61045                 131
             3         Devolución   49611                1532
        D77/88           Dev. ECI   36469                   6
            12      Dev. Muestras   35245                  81
          eDev     Dev. Ecommerce   19406                  13
            10      Dev. Depósito   18161                   3
          RECI Regularización ECI    1498                   1
             8      RepetPdteProd     590                   2
           PER         PERSONA

In [7]:
print("=" * 60)
print("CALIDAD DE CAMPOS CLAVE")
print("=" * 60)
calidad = con.execute("""
    SELECT 
        SUM(CASE WHEN id_cliente IS NULL THEN 1 ELSE 0 END) AS nulos_id_cliente,
        SUM(CASE WHEN fecha_venta IS NULL THEN 1 ELSE 0 END) AS nulos_fecha,
        SUM(CASE WHEN cod_modelo IS NULL OR TRIM(cod_modelo) = '' THEN 1 ELSE 0 END) AS nulos_cod_modelo,
        SUM(CASE WHEN cod_color  IS NULL OR TRIM(cod_color)  = '' THEN 1 ELSE 0 END) AS nulos_cod_color,
        SUM(CASE WHEN cod_serie  IS NULL OR TRIM(cod_serie)  = '' THEN 1 ELSE 0 END) AS nulos_cod_serie,
        SUM(CASE WHEN talla      IS NULL OR TRIM(talla)      = '' THEN 1 ELSE 0 END) AS nulos_talla,
        SUM(CASE WHEN cantidad_neta IS NULL THEN 1 ELSE 0 END) AS nulos_cant_neta,
        SUM(CASE WHEN id_tipo_pedido IS NULL THEN 1 ELSE 0 END) AS nulos_tipo_pedido
    FROM bronze.ventas_minoristas
""").fetchdf()
print(calidad.to_string(index=False))

CALIDAD DE CAMPOS CLAVE
 nulos_id_cliente  nulos_fecha  nulos_cod_modelo  nulos_cod_color  nulos_cod_serie  nulos_talla  nulos_cant_neta  nulos_tipo_pedido
              0.0          0.0               0.0              0.0              0.0          0.0              0.0                0.0


In [8]:
print("=" * 60)
print("ANÁLISIS DE UNICIDAD DEL SKU")
print("=" * 60)

# Opción A: SKU = modelo + color + talla
opt_a = con.execute("""
    SELECT COUNT(DISTINCT (cod_modelo || '-' || cod_color || '-' || talla)) AS combinaciones
    FROM bronze.ventas_minoristas
""").fetchone()[0]

# Opción B: SKU = modelo + color + talla + serie
opt_b = con.execute("""
    SELECT COUNT(DISTINCT (cod_modelo || '-' || cod_color || '-' || talla || '-' || cod_serie)) AS combinaciones
    FROM bronze.ventas_minoristas
""").fetchone()[0]

print(f"Opción A (modelo+color+talla)        : {opt_a:,} SKUs distintos")
print(f"Opción B (modelo+color+talla+serie)  : {opt_b:,} SKUs distintos")
print(f"Diferencia                           : {opt_b - opt_a:,}")

# Detectar combos modelo+color+talla con varias series (si los hay)
print("\n" + "=" * 60)
print("¿La misma combinación modelo+color+talla aparece con varias series?")
print("=" * 60)
multi_serie = con.execute("""
    SELECT COUNT(*) AS combos_con_varias_series
    FROM (
        SELECT cod_modelo, cod_color, talla, COUNT(DISTINCT cod_serie) AS n_series
        FROM bronze.ventas_minoristas
        GROUP BY cod_modelo, cod_color, talla
        HAVING COUNT(DISTINCT cod_serie) > 1
    )
""").fetchone()[0]
print(f"Combos modelo+color+talla con >1 serie: {multi_serie:,}")

ANÁLISIS DE UNICIDAD DEL SKU
Opción A (modelo+color+talla)        : 49,242 SKUs distintos
Opción B (modelo+color+talla+serie)  : 49,242 SKUs distintos
Diferencia                           : 0

¿La misma combinación modelo+color+talla aparece con varias series?
Combos modelo+color+talla con >1 serie: 0


In [9]:
print("=" * 60)
print("DISTRIBUCIÓN DE CANTIDAD_NETA POR TIPO DE PEDIDO")
print("=" * 60)
signo_cant = con.execute("""
    SELECT 
        id_tipo_pedido,
        desc_tipo_pedido,
        SUM(CASE WHEN cantidad_neta > 0 THEN 1 ELSE 0 END) AS filas_positivas,
        SUM(CASE WHEN cantidad_neta < 0 THEN 1 ELSE 0 END) AS filas_negativas,
        SUM(CASE WHEN cantidad_neta = 0 THEN 1 ELSE 0 END) AS filas_cero,
        SUM(cantidad_neta) AS suma_total
    FROM bronze.ventas_minoristas
    GROUP BY id_tipo_pedido, desc_tipo_pedido
    ORDER BY suma_total DESC
""").fetchdf()
print(signo_cant.to_string(index=False))

DISTRIBUCIÓN DE CANTIDAD_NETA POR TIPO DE PEDIDO
id_tipo_pedido   desc_tipo_pedido  filas_positivas  filas_negativas  filas_cero  suma_total
             1          Temporada        1606006.0              0.0         0.0   2648075.0
             2         Repetición         615011.0             68.0         0.0   1307701.0
             7           Depósito         140130.0              5.0         0.0    539168.0
          eCom          Ecommerce         111231.0              0.0         2.0    114825.0
           B2B                B2B          82940.0              0.0         0.0    107203.0
         77/88                ECI          92678.0              0.0         0.0     94283.0
             5           Muestras          60980.0             65.0         0.0     66731.0
             8      RepetPdteProd            588.0              2.0         0.0     17878.0
          RECI Regularización ECI           1498.0              0.0         0.0     11975.0
           N/A                N

## 3. Construcción de `silver.ventas_minoristas`

A partir de los hallazgos de la fase de inspección, se aplican las siguientes transformaciones:

| Transformación | Detalle |
|---|---|
| **Filtro temporal** | Mantener solo registros con `fecha_venta` entre 2022-01-01 y 2025-12-31 (safety check; en bronze ya están filtrados) |
| **Casteo de tipos** | `id_cliente` BIGINT → VARCHAR para coherencia con `dim_cliente` y `fact_lineas_pedido` |
| **Normalización de texto** | `TRIM` y `UPPER` en `cod_modelo`, `cod_color`, `cod_serie`, `talla`, `id_tipo_pedido` |
| **Creación de `id_sku`** | Concatenación `cod_modelo-cod_color-talla` (49.242 SKUs únicos validados) |
| **Creación de `es_devolucion`** | Flag booleano: TRUE si `id_tipo_pedido` ∈ {3, 10, 12, D77/88, eDev, DPer} |
| **Eliminación de columnas redundantes** | `nombre_cliente` (ya en `dim_cliente`) |
| **Columnas conservadas** | Todas las demás, incluyendo `id_temporada`, `cantidad_ventas_sin_devoluciones`, descripciones |

In [10]:
print("Construyendo silver.ventas_minoristas... (puede tardar 30-60 s, son 2,87 M filas)\n")

con.execute("""
    CREATE OR REPLACE TABLE silver.ventas_minoristas AS
    SELECT
        -- Fecha
        fecha_venta,
        
        -- Cliente (casteado a VARCHAR para coherencia entre tablas)
        CAST(id_cliente AS VARCHAR) AS id_cliente,
        
        -- Producto: códigos normalizados
        UPPER(TRIM(cod_modelo))  AS cod_modelo,
        UPPER(TRIM(cod_color))   AS cod_color,
        UPPER(TRIM(cod_serie))   AS cod_serie,
        UPPER(TRIM(talla))       AS talla,
        
        -- Producto: descripciones (solo TRIM, conservan capitalización original)
        TRIM(desc_modelo) AS desc_modelo,
        TRIM(desc_color)  AS desc_color,
        TRIM(desc_serie)  AS desc_serie,
        
        -- SKU derivado
        UPPER(TRIM(cod_modelo)) || '-' || UPPER(TRIM(cod_color)) || '-' || UPPER(TRIM(talla)) AS id_sku,
        
        -- Tipo de pedido (canal)
        UPPER(TRIM(id_tipo_pedido)) AS id_tipo_pedido,
        TRIM(desc_tipo_pedido)      AS desc_tipo_pedido,
        
        -- Flag de devolución (basado en tipo de pedido, no en signo)
        CASE 
            WHEN UPPER(TRIM(id_tipo_pedido)) IN ('3', '10', '12', 'D77/88', 'EDEV', 'DPER') 
            THEN TRUE 
            ELSE FALSE 
        END AS es_devolucion,
        
        -- Cantidades
        cantidad_neta,
        cantidad_ventas_sin_devoluciones,
        
        -- Temporada
        id_temporada
        
    FROM bronze.ventas_minoristas
    WHERE fecha_venta BETWEEN DATE '2022-01-01' AND DATE '2025-12-31'
""")

print("✅ silver.ventas_minoristas creada correctamente")

Construyendo silver.ventas_minoristas... (puede tardar 30-60 s, son 2,87 M filas)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ silver.ventas_minoristas creada correctamente


## 4. Validación de la tabla resultante

In [11]:
print("=" * 60)
print("AUDITORÍA DE VOLÚMENES: BRONZE → SILVER")
print("=" * 60)

n_bronze = con.execute("SELECT COUNT(*) FROM bronze.ventas_minoristas").fetchone()[0]
n_silver = con.execute("SELECT COUNT(*) FROM silver.ventas_minoristas").fetchone()[0]

print(f"Filas en bronze : {n_bronze:>12,}")
print(f"Filas en silver : {n_silver:>12,}")
print(f"Diferencia      : {n_bronze - n_silver:>12,}")
print(f"% conservado    : {n_silver / n_bronze * 100:>12.2f} %")

AUDITORÍA DE VOLÚMENES: BRONZE → SILVER
Filas en bronze :    2,870,570
Filas en silver :    2,870,570
Diferencia      :            0
% conservado    :       100.00 %


In [12]:
print("=" * 60)
print("ESQUEMA DE silver.ventas_minoristas")
print("=" * 60)
esquema_silver = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'ventas_minoristas'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema_silver.to_string(index=False))

ESQUEMA DE silver.ventas_minoristas
                     column_name data_type
                     fecha_venta      DATE
                      id_cliente   VARCHAR
                      cod_modelo   VARCHAR
                       cod_color   VARCHAR
                       cod_serie   VARCHAR
                           talla   VARCHAR
                     desc_modelo   VARCHAR
                      desc_color   VARCHAR
                      desc_serie   VARCHAR
                          id_sku   VARCHAR
                  id_tipo_pedido   VARCHAR
                desc_tipo_pedido   VARCHAR
                   es_devolucion   BOOLEAN
                   cantidad_neta    BIGINT
cantidad_ventas_sin_devoluciones    BIGINT
                    id_temporada    BIGINT


In [13]:
print("=" * 60)
print("DISTRIBUCIÓN TEMPORAL FINAL (silver)")
print("=" * 60)
temp = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes,
        COUNT(DISTINCT id_sku) AS skus_distintos,
        SUM(cantidad_neta) AS suma_cantidad_neta
    FROM silver.ventas_minoristas
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(temp.to_string(index=False))

DISTRIBUCIÓN TEMPORAL FINAL (silver)
 anio  filas  clientes  skus_distintos  suma_cantidad_neta
 2022 729665      2266           27839           1256163.0
 2023 719482      2232           24396           1186289.0
 2024 705210      2200           21466           1119201.0
 2025 716213      2125           24463           1143147.0


In [14]:
print("=" * 60)
print("VALIDACIÓN DEL FLAG es_devolucion")
print("=" * 60)
val_dev = con.execute("""
    SELECT 
        es_devolucion,
        COUNT(*) AS filas,
        SUM(cantidad_neta) AS suma_cantidad,
        ROUND(AVG(cantidad_neta), 2) AS media_cantidad
    FROM silver.ventas_minoristas
    GROUP BY es_devolucion
    ORDER BY es_devolucion
""").fetchdf()
print(val_dev.to_string(index=False))

VALIDACIÓN DEL FLAG es_devolucion
 es_devolucion   filas  suma_cantidad  media_cantidad
         False 2711667      4909304.0            1.81
          True  158903      -204504.0           -1.29


In [15]:
print("=" * 60)
print("INTEGRIDAD REFERENCIAL CON silver.dim_cliente")
print("=" * 60)
integ = con.execute("""
    SELECT 
        COUNT(DISTINCT v.id_cliente) AS clientes_en_ventas,
        COUNT(DISTINCT d.id_cliente) AS clientes_en_dim,
        COUNT(DISTINCT CASE WHEN d.id_cliente IS NULL THEN v.id_cliente END) AS clientes_huerfanos
    FROM silver.ventas_minoristas v
    LEFT JOIN silver.dim_cliente d ON v.id_cliente = d.id_cliente
""").fetchdf()
print(integ.to_string(index=False))

INTEGRIDAD REFERENCIAL CON silver.dim_cliente
 clientes_en_ventas  clientes_en_dim  clientes_huerfanos
               3054             3051                   3


In [16]:
print("=" * 60)
print("TOP 10 SKUs POR CANTIDAD VENDIDA (excluyendo devoluciones)")
print("=" * 60)
top_skus = con.execute("""
    SELECT 
        id_sku,
        desc_modelo,
        desc_color,
        talla,
        SUM(cantidad_neta) AS unidades_vendidas
    FROM silver.ventas_minoristas
    WHERE es_devolucion = FALSE
    GROUP BY id_sku, desc_modelo, desc_color, talla
    ORDER BY unidades_vendidas DESC
    LIMIT 10
""").fetchdf()
print(top_skus.to_string(index=False))

TOP 10 SKUs POR CANTIDAD VENDIDA (excluyendo devoluciones)
      id_sku                   desc_modelo desc_color talla  unidades_vendidas
 10630-035-M       SUJETADOR SIN AROS 24-H MAQUILLAJE     M            16117.0
002008-185-M      CINTA LIFT-TAPE ADHESIVA       PIEL     M            14605.0
 10630-035-L       SUJETADOR SIN AROS 24-H MAQUILLAJE     L            14405.0
 10630-004-M       SUJETADOR SIN AROS 24-H      NEGRO     M            12314.0
 10615-035-M SUJETADOR ESCOTE PROFUNDO 24H MAQUILLAJE     M            12125.0
 10630-004-L       SUJETADOR SIN AROS 24-H      NEGRO     L            11545.0
10630-035-XL       SUJETADOR SIN AROS 24-H MAQUILLAJE    XL            10994.0
 10615-035-L SUJETADOR ESCOTE PROFUNDO 24H MAQUILLAJE     L             9622.0
 10630-035-S       SUJETADOR SIN AROS 24-H MAQUILLAJE     S             9403.0
 10615-004-M SUJETADOR ESCOTE PROFUNDO 24H      NEGRO     M             9055.0


In [17]:
print("=" * 60)
print("INVESTIGACIÓN DE LOS 3 CLIENTES HUÉRFANOS")
print("=" * 60)
huerfanos = con.execute("""
    SELECT 
        v.id_cliente,
        COUNT(*) AS filas_ventas,
        SUM(v.cantidad_neta) AS unidades,
        MIN(v.fecha_venta) AS primera_venta,
        MAX(v.fecha_venta) AS ultima_venta,
        COUNT(DISTINCT v.desc_tipo_pedido) AS canales_distintos,
        STRING_AGG(DISTINCT v.desc_tipo_pedido, ', ') AS canales
    FROM silver.ventas_minoristas v
    LEFT JOIN silver.dim_cliente d ON v.id_cliente = d.id_cliente
    WHERE d.id_cliente IS NULL
    GROUP BY v.id_cliente
    ORDER BY filas_ventas DESC
""").fetchdf()
print(huerfanos.to_string(index=False))

INVESTIGACIÓN DE LOS 3 CLIENTES HUÉRFANOS
id_cliente  filas_ventas  unidades primera_venta ultima_venta  canales_distintos    canales
     32211           225     526.0    2025-09-16   2025-09-16                  1  Temporada
      4753            40      40.0    2022-05-31   2022-06-01                  1 Repetición
      2443             8      12.0    2023-06-28   2023-06-28                  1 Repetición


## 5. Conclusiones del notebook 05

### Resumen del proceso

La tabla `silver.ventas_minoristas` se ha construido a partir de `bronze.ventas_minoristas` aplicando transformaciones de tipo (casteo de `id_cliente` a VARCHAR), normalización de texto en códigos de producto y canal, creación del identificador derivado `id_sku`, y construcción del flag `es_devolucion` basado en el tipo de pedido. Se ha conservado la totalidad del volumen (2.870.570 filas, 100%), ya que la ventana temporal 2022-2025 ya estaba aplicada en bronze de origen.

### Hallazgos principales

**Calidad de datos: excelente.** Cero valores nulos en todos los campos clave (cliente, fecha, producto, cantidad, tipo de pedido). No se han detectado fechas anómalas dentro de la ventana de análisis.

**Estructura del SKU validada con 3 componentes.** El análisis de unicidad ha confirmado que la combinación `cod_modelo + cod_color + talla` produce 49.242 SKUs distintos, idéntico número al obtenido añadiendo `cod_serie`. Esto demuestra que la serie es una agrupación de nivel superior (probablemente categoría comercial) y no es necesaria para identificar el SKU. Se conserva como columna independiente para análisis posteriores de mix de producto.

**Distribución por canal (`desc_tipo_pedido`) muy heterogénea.** El 77% de las operaciones corresponde a pedidos B2B (Temporada con 56% + Repetición con 21%), distribuidos entre ~2.500 clientes. Los canales digitales y corporativos (Ecommerce, Depósito, ECI) concentran el 12% de las filas pero solo 5-8 clientes únicos cada uno: cada plataforma figura como una cuenta corporativa única, no como consumidores finales individuales. Esta característica condiciona el diseño del clustering posterior: los clientes B2B reales y los clientes "macro" de canales digitales no son comparables y deberán tratarse por separado.

**Devoluciones identificadas y trazables.** 158.903 filas (5,5% del total) corresponden a devoluciones, identificadas por `id_tipo_pedido` ∈ {3, 10, 12, D77/88, EDEV, DPER}. La validación cruzada confirma que estas filas tienen `cantidad_neta` mayoritariamente negativa (>99,9%) y suman -204.504 unidades. El flag `es_devolucion` permite filtrarlas o analizarlas en notebooks posteriores según la pregunta de negocio.

**Integridad referencial casi completa.** 3.051 clientes de `silver.ventas_minoristas` (99,9%) cruzan correctamente con `silver.dim_cliente`. Los 3 clientes huérfanos representan 273 filas (0,01%) y corresponden a operaciones puntuales de 1-2 días, probablemente clientes dados de alta para campañas específicas y aún sin maestro consolidado. Se asumirán como `NULL` en los datos maestros al construir `gold.cliente_360`.

### Cuestiones pendientes con el tutor

1. **Interpretación de `Temporada` vs `Repetición`**: ¿son canales independientes o son fases distintas del mismo proceso B2B (campaña inicial vs reposiciones de stock)? Condiciona la categorización en `tipo_cliente` de la tabla cliente final.
2. **Clientes "macro" de canales digitales**: confirmar que Ecommerce, Depósito, ECI y similares se registran en el ERP como cuentas corporativas únicas, no como consumidores finales individuales.

### Próximo notebook

`06_silver_tiempo.ipynb` — Construcción de la tabla calendario para análisis temporal: días, meses, trimestres, años, festivos nacionales españoles, periodos de rebajas, San Valentín, Black Friday y temporadas de moda (PV/OI).

In [18]:
con.close()
print("✅ Conexión cerrada. silver.tiempo guardada en disco.")

✅ Conexión cerrada. silver.tiempo guardada en disco.
